# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [3]:
removed = int(df.duplicated().sum())
df = df.drop_duplicates()
log('1', 'dropped exact duplicate rows', removed)
df.head()

[1] dropped exact duplicate rows (15 row(s))


,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 2 — clean `price` -> float

In [4]:
df['price'] = (df['price'].astype(str).str.replace(r'[$,\s]', '', regex=True).astype(float))

assert df['price'].dtype == float
log('2', 'cleaned price into type float', len(df))

[2] cleaned price into type float (300 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [5]:
df['qty'] = pd.to_numeric(df['qty'], errors='coerce')

missing = int(df['qty'].isna().sum())
negative = int((df['qty'] < 0).sum())
df = df[df['qty'].notna() & (df['qty'] >= 0)].copy()
df['qty'] = df['qty'].astype(int)

log('3', 'dropped rows with missing quantity', missing)
log('3', 'dropped rows with negative quantity', negative)

[3] dropped rows with missing quantity (12 row(s))
[3] dropped rows with negative quantity (13 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [6]:
print(df['item'].value_counts())

orig = df['item'].copy()

df['item'] = (df['item'].str.lower().str.strip().str.replace(r'[^a-z0-9]', '', regex=True))

ITEM_MAP = {
    'foamfinger':     'Foam Finger',
    'cheeseburger':    'Cheeseburger',
    'rainponcho':     'Rain Poncho'
}
df['item'] = df['item'].map(ITEM_MAP)

unmapped_item = df['item'].isna().sum()

log('4', 'normalized item case/punctuation and mapped to labels', int((orig != df['item']).sum()))

print(df['item'].value_counts())

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[4] normalized item case/punctuation and mapped to labels (126 row(s))
item
Foam Finger     97
Rain Poncho     91
Cheeseburger    87
Name: count, dtype: int64


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [7]:
print(df['category'].value_counts())


df['category'] = (df['category'].str.lower().str.strip().str.replace(r'[^a-z0-9]', '', regex=True))

CATEGORY_MAP = {
    'food':     'Food',
    'merch':    'Merch',
    'apparel':  'Apparel',
    'raingear': 'Rain Gear',
}
df['category'] = df['category'].map(CATEGORY_MAP)

unmapped_cat = df['category'].isna().sum()

log('5', 'normalized category case/punctuation and mapped to canonical labels', int((orig != df['category']).sum()))

print(df['category'].value_counts())

category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
[5] normalized category case/punctuation and mapped to canonical labels (275 row(s))
category
Food         95
Rain Gear    86
Merch        51
Apparel      43
Name: count, dtype: int64


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [8]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
assert unmapped_item == 0, f'{unmapped_item} categories not in ITEM_MAP'
assert unmapped_cat == 0, f'{unmapped_cat} categories not in CATEGORY_MAP'
print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [9]:
df['revenue'] = df['qty'] * df['price']

by_cat = df.groupby('item')['revenue'].sum().sort_values(ascending=False).round(2)
print(by_cat)
print('TOTAL', round(df['revenue'].sum(), 2))

item
Rain Poncho     1729.5
Foam Finger     1593.0
Cheeseburger    1417.5
Name: revenue, dtype: float64
TOTAL 4740.0


**What I would tell the vendor:** I would tell the vendor to stock more Rain Poncho, as they are the item that generates the most revenue.

### TODO 8 — read back your log

In [10]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,1,dropped exact duplicate rows,15
1,2,cleaned price into type float,300
2,3,dropped rows with missing quantity,12
3,3,dropped rows with negative quantity,13
4,4,normalized item case/punctuation and mapped to...,126
5,5,normalized category case/punctuation and mappe...,275


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

a) Removing the duplicate rows changed the total most, as revenue dropped from 4,852.50 to 4,594.50, which is 258. Dropping the negative rows also changed revenue significantly, but not as much as the duplicates.

b) A reasonable person might've not dropped the negative and missing quanitities. I dropped them as invalid entries, which gives 4,740.00, but you could also treat them as returns or refunds and keep them. Then, revenue would be 4,594.50, or 145.50 less. I dropped them because nothing in the data marks them as returns. There's no return flag, no link back to an original order, and every order_id is unique after deduplication, so none of them offsets an earlier sale.